# Split Test Analysis 

# NOTE (Before Running Our Stratified SPLIT)

Reasons why we don't use SMOTE is due to the nature of our data. Each applicant has previous application data reflecting information regarding their payments/ their ext_sources (Credit Score), employment status, etc. Some of this information is missing since some applicants may have not applied previously or haven't are applying since a long time. This creates a problem with smote because trying to create data to balance people who struggle or don't struggle may contain this issue. Meaning, we are going to have to create excessive data that will try to account for the missing data that each user is missing. This is a problem due to subsampling and imbalanced nature of our data. What is important to address this problem is by ensuring our hyperparameters in all of our models including our NN to be able to address this issue. 


## Addressing Class Imbalance

Each model handles imbalance by weighting rows, so that errors on the minority
class cost more than errors on the majority class.

### 1. Class weights

**Random Forest** (`class_weight="balanced"`):

$$w_c = \frac{N}{K \cdot n_c}$$

- $N$ = total number of rows
- $K$ = number of classes
- $n_c$ = number of rows in class $c$

**LightGBM / XGBoost** (`scale_pos_weight`):

$$s = \frac{n_{neg}}{n_{pos}}$$

Positive rows get weight $w_i = s$, negative rows get weight $w_i = 1$.

### 2. Random Forest: weighted Gini impurity

Class proportions in a node use weight sums instead of row counts:

$$p_k = \frac{\sum_{i \in \text{node},\ y_i = k} w_i}{\sum_{i \in \text{node}} w_i}
\qquad
\text{Gini} = 1 - \sum_{k} p_k^2$$

### 3. LightGBM / XGBoost: weighted log loss

$$L = -\sum_{i} w_i \left[ y_i \log(p_i) + (1 - y_i) \log(1 - p_i) \right]$$

The weight carries into the gradient and hessian used to build each tree:

$$g_i = w_i (p_i - y_i)
\qquad
h_i = w_i \, p_i (1 - p_i)$$

Leaf values and split gain are computed from their sums
($G = \sum g_i$, $H = \sum h_i$):

$$\text{leaf value} = -\frac{G}{H + \lambda}$$

$$\text{gain} = \frac{1}{2} \left[ \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{G^2}{H + \lambda} \right]$$

### 4. Effect

With a class ratio of $s$, a misclassified positive contributes $s$ times as
much to the loss as a misclassified negative, so both classes carry equal
total weight during training.

**Note:** weighting inflates predicted probabilities for the positive class.
Ranking metrics such as ROC AUC are largely unaffected, but the probabilities
are no longer calibrated to the true class frequencies.


In [5]:
import duckdb
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from home_credit_neural_network import HomeCreditNN, prepare_features

In [6]:
# Loader Function Needed to fix
train_merged = pd.read_parquet("data/processed/train_merged.parquet")

X, y = prepare_features(train_merged)
cat_cols = X.select_dtypes(include=["object", "str"]).columns.to_list()
num_col = [c for c in X.columns if c not in cat_cols]
X[cat_cols] = X[cat_cols].fillna("MISSING")

print(X.shape, f"{len(num_col)} numeric, {len(cat_cols)} categorical")

(307511, 173) 157 numeric, 16 categorical


In [7]:
# Helper Functions 
def loader_split(name):
    
    # This will utilize duckdb to read the files within the data_splits
    s = duckdb.sql(f"SELECT SK_ID_CURR, split FROM 'data_splits/{name}.parquet'").df()
        
    # match by client ID, not row order instead of loading the full dataset from parquet
    split = train_merged["SK_ID_CURR"].map(s.set_index("SK_ID_CURR")["split"]) 
        
    mask = {k: (split == k).to_numpy() for k in ["train", "dev", "validation"]}
    for k, m in mask.items():
        print(f"{k:<11}, {m.sum():,} clients default rate{y[m].mean():.2%}")
    return mask #data based of the data split parquet

def to_category(X, train_mask):
    # categories learned from the train rows only, applied to every row
    X_cat = X.copy()
    for col in cat_cols:
        # essentially this code is verifying the categories are present with the data set. 
        # using our train_merged data set we are able to compare and contrast whether this is present or not
        #[Rows, cols] essentially shows us what we want to keep
        cats = pd.Categorical(X.loc[train_mask, col]).categories # this pulls a list of categories 

        # anything not inside of cat is turned into NAN
        X_cat[col] = pd.Categorical(X[col], categories=cats)
    return X_cat

def run_models(split_name):
    m = loader_split(split_name)
    tr, dv, va, = m["train"], m["dev"], m["validation"]
    X_tree = to_category(X, tr)
    results, val_preds = [], {}

    def record(model_name, p_train, p_val, n_iter=None):
        val_preds[model_name] = p_val
        results.append({
            "split_file": split_name,
            "model": model_name,
            "train_auc": roc_auc_score(y[tr], p_train),
            "val_auc": roc_auc_score(y[va], p_val),
            "val_pr_auc": average_precision_score(y[va], p_val),
            "iterations": n_iter
        })
        print(f"{model_name}, validation AUC {results[-1]["val_auc"]}")

    """
    Notes on Pipeline for RANDOM FORREST... 
    
    **SimpleImputer** : fillls in any NAN values with the median 
    **OrdinalEncoder**: encodes the categorical variables for the models to inetpret them. Morever, fills in any NAN values with -1 
    
    Hyperparameters:
    class_weight="balanced_subsample" -- important to use because it ensure that the model accounts for the imbalance within the data
    
    """
    rf = Pipeline([
        ("prep", ColumnTransformer([
            ("num", SimpleImputer(strategy="median", add_indicator=True), num_col),
            ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols),
        ], verbose_feature_names_out=False)),
        
        ("rf", RandomForestClassifier(
            n_estimators=300, criterion="entropy", max_features="sqrt", min_samples_leaf=50,
            max_samples=0.5, class_weight="balanced_subsample", n_jobs=-1, random_state=420,
        )),
    ])
    rf.fit(X[tr], y[tr])
    record("Random Forest", rf.predict_proba(X[tr])[:, 1], rf.predict_proba(X[va])[:, 1])

    """
    Notes on LightGBM...
    X_tree ---> uses to_category to encode and keep variables that are needed for the LightGBM classifier to run
    
    Hyperparameters: 
    class_weight='balanced' --- will modify the penalization of the weights to handle imbalanced classfiers in the data
    
    **REFER TO EQUATION ABOVE**
       
    """
    lgbm = lgb.LGBMClassifier(
        n_estimators=3000, learning_rate=0.03, num_leaves=31, min_child_samples=100,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=1.0,
        class_weight="balanced", random_state=420, n_jobs=-1, verbose=-1
    )
    lgbm.fit(X_tree[tr], y[tr], eval_set=[(X_tree[dv], y[dv])], eval_metric="auc",
             callbacks=[lgb.early_stopping(100, verbose=False)])
    record("LightGBM", lgbm.predict_proba(X_tree[tr])[:, 1], lgbm.predict_proba(X_tree[va])[:, 1], lgbm.best_iteration_)

    """
    Notes on XGBOOST...
    X_tree ---> uses to_category to encode and keep variables that are needed for the LightGBM classifier to run
    
    Hyperparameters:
    
    """
    
    xgb_model = xgb.XGBClassifier(
        n_estimators=3000, learning_rate=0.03, max_depth=6, min_child_weight=10,
        subsample=0.8, colsample_bytree=0.6, reg_lambda=1.0,
        tree_method="hist", enable_categorical=True, max_cat_to_onehot=1,
        eval_metric="auc", early_stopping_rounds=100, n_jobs=-1, random_state=420
    )
    xgb_model.fit(X_tree[tr], y[tr], eval_set=[(X_tree[dv], y[dv])], verbose=False)
    record("XGBoost", xgb_model.predict_proba(X_tree[tr])[:, 1], xgb_model.predict_proba(X_tree[va])[:, 1], xgb_model.best_iteration)

    """
    Notes on Neural Network
    
    The Neural Network's hyper parameters and it's architecture are in a Read_me.md
    
    """

    nn_model = HomeCreditNN(class_weight="balanced", verbose=False)
    nn_model.fit(X[tr], y[tr], X_val=X[dv], y_val=y[dv])
    record("Neural Network", nn_model.predict_proba(X[tr])[:, 1], nn_model.predict_proba(X[va])[:, 1], len(nn_model.history_))

    return pd.DataFrame(results), val_preds, m

# Non Random Split

#### Note:
This data is grabbing data from the test dataset provided from kaggle and grabs data that is most simlar to our test set. By doing this we hope that our model will be trained to handle similar features as that of our test set

In [8]:
results_non_random, preds_non_random, masks_non_random = run_models("non_random")
results_non_random

train      , 215,257 clients default rate8.33%
dev        , 46,127 clients default rate7.19%
validation , 46,127 clients default rate7.75%
Random Forest, validation AUC 0.759139138311037


c:\Users\Flami\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


LightGBM, validation AUC 0.7800559581167426
XGBoost, validation AUC 0.7823546176750027
Neural Network, validation AUC 0.7721953125915964


,split_file,model,train_auc,val_auc,val_pr_auc,iterations
0,non_random,Random Forest,0.869730,0.759139,0.229975,NaN
1,non_random,LightGBM,0.864637,0.780056,0.256837,667.0
2,non_random,XGBoost,0.893215,0.782355,0.256677,1003.0
3,non_random,Neural Network,0.799266,0.772195,0.245007,11.0


# Random 

Note: 

Just a normal random split dataset...

In [9]:
results_random, preds_random, masks_random = run_models("random")
results_random

train      , 215,257 clients default rate8.09%
dev        , 46,127 clients default rate8.05%
validation , 46,127 clients default rate8.03%
Random Forest, validation AUC 0.7587917920697598


c:\Users\Flami\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


LightGBM, validation AUC 0.7835808078002103
XGBoost, validation AUC 0.7842746284579286
Neural Network, validation AUC 0.7750545379645137


,split_file,model,train_auc,val_auc,val_pr_auc,iterations
0,random,Random Forest,0.872927,0.758792,0.232416,NaN
1,random,LightGBM,0.865676,0.783581,0.266953,647.0
2,random,XGBoost,0.877096,0.784275,0.269213,748.0
3,random,Neural Network,0.813618,0.775055,0.256526,13.0


# Stratified Target

Note:

Stratified the data onto our target variable

In [10]:
results_strat, preds_strat, masks_strat = run_models("stratified_target")
results_strat

train      , 215,259 clients default rate8.07%
dev        , 46,125 clients default rate8.07%
validation , 46,127 clients default rate8.07%
Random Forest, validation AUC 0.759905453510841


c:\Users\Flami\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


LightGBM, validation AUC 0.7825578809516673
XGBoost, validation AUC 0.7838932659168548
Neural Network, validation AUC 0.7736030111107444


,split_file,model,train_auc,val_auc,val_pr_auc,iterations
0,stratified_target,Random Forest,0.873243,0.759905,0.232465,NaN
1,stratified_target,LightGBM,0.877391,0.782558,0.270119,777.0
2,stratified_target,XGBoost,0.899389,0.783893,0.268466,1065.0
3,stratified_target,Neural Network,0.808929,0.773603,0.256317,13.0


# Stratified Target (Education)

Note:

Stratified the data based of education

In [11]:
results_strat_edu, preds_strat_edu, masks_strat_edu = run_models("stratified_target_edu")
results_strat_edu

train      , 215,262 clients default rate8.07%
dev        , 46,122 clients default rate8.07%
validation , 46,127 clients default rate8.07%
Random Forest, validation AUC 0.7616021926888267


c:\Users\Flami\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


LightGBM, validation AUC 0.7828072654507122
XGBoost, validation AUC 0.7832590231276069
Neural Network, validation AUC 0.7740905014447202


,split_file,model,train_auc,val_auc,val_pr_auc,iterations
0,stratified_target_edu,Random Forest,0.873219,0.761602,0.246887,NaN
1,stratified_target_edu,LightGBM,0.868338,0.782807,0.281333,672.0
2,stratified_target_edu,XGBoost,0.880761,0.783259,0.279630,792.0
3,stratified_target_edu,Neural Network,0.806203,0.774091,0.272061,12.0


# Visualization

In [12]:
all_results = pd.concat([results_non_random, results_random, results_strat, results_strat_edu], ignore_index=True)

print("Validation ROC AUC")
print(all_results.pivot(index="model", columns="split_file", values="val_auc").round(4))
print("\nValidation PR AUC")
print(all_results.pivot(index="model", columns="split_file", values="val_pr_auc").round(4))


Validation ROC AUC
split_file      non_random  random  stratified_target  stratified_target_edu
model                                                                       
LightGBM            0.7801  0.7836             0.7826                 0.7828
Neural Network      0.7722  0.7751             0.7736                 0.7741
Random Forest       0.7591  0.7588             0.7599                 0.7616
XGBoost             0.7824  0.7843             0.7839                 0.7833

Validation PR AUC
split_file      non_random  random  stratified_target  stratified_target_edu
model                                                                       
LightGBM            0.2568  0.2670             0.2701                 0.2813
Neural Network      0.2450  0.2565             0.2563                 0.2721
Random Forest       0.2300  0.2324             0.2325                 0.2469
XGBoost             0.2567  0.2692             0.2685                 0.2796
